# 🗂️ Notebook 2: Shopping Cart — Data Model & APIs

## 🛠️ Setup

```bash
cd 06-system-designs/shopping-cart
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Entities

- **Product** — SKU + price + stock.
- **Cart** — belongs to a user; has line items.
- **Reservation** — short-lived lock on N units of a SKU.
- **Order** — paid, immutable record.

## Pydantic models

We use `pydantic` for data validation — it forces us to think about types, required fields, and invariants up front.

In [ ]:
from decimal import Decimal
from typing import Literal
from pydantic import BaseModel, Field, field_validator

class Product(BaseModel):
    sku: str
    name: str
    price: Decimal
    stock: int = Field(ge=0)

class CartItem(BaseModel):
    sku: str
    qty: int = Field(ge=1)

class Cart(BaseModel):
    user_id: int
    items: list[CartItem] = []

    def total(self, catalog: dict[str, Product]) -> Decimal:
        return sum((catalog[i.sku].price * i.qty for i in self.items), Decimal(0))

class Order(BaseModel):
    id: int
    user_id: int
    items: list[CartItem]
    status: Literal["pending","paid","failed","refunded"] = "pending"

cat = {"A": Product(sku="A", name="Book", price=Decimal("10"), stock=100)}
c = Cart(user_id=1, items=[CartItem(sku="A", qty=3)])
print("total:", c.total(cat))

## HTTP APIs

| Method | Path | What |
|---|---|---|
| GET | `/cart` | Current user's cart |
| POST | `/cart/items` | Add item |
| DELETE | `/cart/items/{sku}` | Remove |
| POST | `/checkout` (with Idempotency-Key) | Reserve + charge |
| GET | `/orders/{id}` | Order status |


## Quick demo

In [ ]:
# Idempotency keys: same key → same result (no double charge)
seen = {}  # key → response

def checkout(idem_key, cart_total):
    if idem_key in seen:
        print("replay → returning cached response")
        return seen[idem_key]
    order = {"id": len(seen)+1, "total": cart_total, "status": "paid"}
    seen[idem_key] = order
    return order

print(checkout("abc", 30))
print(checkout("abc", 30))   # same key, same response
print(checkout("xyz", 30))   # new order

## Takeaways

- Small, typed models make the service boundary crisp.
- Public APIs hide internal IDs and expose human-friendly resources.
- Write one happy-path test per endpoint before scaling out.